# S12: Integración de datos de NovaMarket

**Responsables:** Daniel Pallares, Arbelaez y Sanchez

Este flujo integra el dataset transformado de S11 con el CRM de clientes y el catálogo de bodegas. Se conserva una fila por pedido y se registran indicadores de coincidencia para hacer auditable la cobertura.

> **Estado: PENDIENTE DE REVISIÓN INDEPENDIENTE.** Este resultado no se declara listo hasta que otro analista valide las llaves, la cobertura y el tratamiento de las categorías sin bodega asignada.

Entradas: `../S11/S11_PD_PallaresArbelaezSanchez_DatasetTransformado.csv`, `../NovaMarket_datos_crudos.csv`, `bodega_categorias_novamarket.csv` y `crm_clientes_novamarket.csv`. La categoría se recupera desde el origen por `id_pedido`, porque S11 conserva sus variables codificadas pero no la columna categórica original.

## 1. Configuración y lectura de fuentes

In [1]:
from pathlib import Path
import unicodedata
import pandas as pd

WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / 'NovaMarket_datos_crudos.csv').exists():
    WORKSPACE = WORKSPACE.parent
S12_DIR = WORKSPACE / 'S12'

pedidos = pd.read_csv(
    WORKSPACE / 'S11' / 'S11_PD_PallaresArbelaezSanchez_DatasetTransformado.csv',
    encoding='utf-8-sig',
    dtype={'id_pedido': 'string'},
)
origen = pd.read_csv(
    WORKSPACE / 'NovaMarket_datos_crudos.csv',
    usecols=['id_pedido', 'categoria_producto'],
    encoding='utf-8-sig',
    dtype={'id_pedido': 'string'},
)
crm = pd.read_csv(S12_DIR / 'crm_clientes_novamarket.csv', encoding='utf-8-sig')
bodegas = pd.read_csv(S12_DIR / 'bodega_categorias_novamarket.csv', encoding='utf-8-sig')

def normalizar_llave(valores):
    def normalizar_valor(valor):
        if pd.isna(valor) or not str(valor).strip():
            return pd.NA
        texto = unicodedata.normalize('NFKD', str(valor).strip().casefold())
        return ''.join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return valores.map(normalizar_valor).astype('string')

print(f'Pedidos S11: {len(pedidos):,} filas; {pedidos.id_pedido.nunique():,} llaves únicas')
print(f'Filas fuente cruda: {len(origen):,}; CRM: {len(crm):,}; categorías de bodega: {len(bodegas):,}')

Pedidos S11: 551,790 filas; 551,790 llaves únicas
Filas fuente cruda: 640,000; CRM: 411,840; categorías de bodega: 7


## 2. Validación de llaves y recuperación de categoría
Se normalizan espacios, mayúsculas/minúsculas y tildes para comparar, sin alterar los valores originales del dataset. Antes de cruzar, se exige unicidad de las dimensiones y consistencia de la categoría por pedido.

In [ ]:
assert pedidos['id_pedido'].notna().all(), 'Hay pedidos sin id_pedido en S11.'
assert pedidos['id_pedido'].is_unique, 'La llave id_pedido no es única en S11.'

categorias_por_pedido = origen.groupby('id_pedido')['categoria_producto'].nunique(dropna=False)
assert categorias_por_pedido.le(1).all(), 'Un pedido tiene más de una categoría en el origen.'
contexto_pedido = origen.drop_duplicates('id_pedido').copy()
assert contexto_pedido['id_pedido'].is_unique
assert pedidos['id_pedido'].isin(contexto_pedido['id_pedido']).all(), 'Hay pedidos S11 que no existen en el origen.'

crm['email_key'] = normalizar_llave(crm['email_cliente'])
bodegas['categoria_key'] = normalizar_llave(bodegas['categoria'])
assert crm['email_key'].notna().all(), 'El CRM contiene emails vacíos.'
assert crm['email_key'].is_unique, 'El email normalizado no es único en CRM.'
assert bodegas['categoria_key'].notna().all(), 'El catálogo contiene categorías vacías.'
assert bodegas['categoria_key'].is_unique, 'La categoría normalizada no es única en bodegas.'

filas_pedidos = len(pedidos)
integrado = pedidos.merge(
    contexto_pedido[['id_pedido', 'categoria_producto']],
    on='id_pedido', how='left', validate='one_to_one'
)
integrado['email_key'] = normalizar_llave(integrado['correo_cliente'])
crm_dimension = crm[[
    'email_key', 'ciudad_residencia', 'fecha_inscripcion_club',
    'puntos_fidelidad_actuales', 'segmento_crm', 'acepta_promociones'
]]
integrado = integrado.merge(
    crm_dimension, on='email_key', how='left', validate='many_to_one', indicator='_crm_join'
)
integrado['crm_encontrado'] = integrado.pop('_crm_join').eq('both')
integrado = integrado.drop(columns='email_key')

integrado['categoria_key'] = normalizar_llave(integrado['categoria_producto'])
bodega_dimension = bodegas[[
    'categoria_key', 'proveedor_principal', 'bodega_asignada', 'dias_reposicion', 'stock_minimo'
]]
integrado = integrado.merge(
    bodega_dimension, on='categoria_key', how='left', validate='many_to_one', indicator='_bodega_join'
)
integrado['bodega_encontrada'] = integrado.pop('_bodega_join').eq('both')
integrado = integrado.drop(columns='categoria_key')
integrado['estado_revision_independiente'] = 'PENDIENTE_REVISION_ANALISTA'

assert len(integrado) == filas_pedidos, 'La integración cambió el número de pedidos.'
assert integrado['id_pedido'].is_unique, 'La integración produjo ids de pedido duplicados.'
assert integrado.duplicated().sum() == 0, 'Se detectaron duplicados exactos después de integrar.'

resumen = pd.Series({
    'pedidos integrados': len(integrado),
    'ids de pedido únicos': integrado['id_pedido'].nunique(),
    'coincidencias CRM': int(integrado['crm_encontrado'].sum()),
    'sin coincidencia CRM': int((~integrado['crm_encontrado']).sum()),
    'coincidencias catálogo bodega': int(integrado['bodega_encontrada'].sum()),
    'sin bodega asignada': int((~integrado['bodega_encontrada']).sum()),
    'duplicados exactos integrados': int(integrado.duplicated().sum()),
})
display(resumen.to_frame('valor'))

categorias_sin_bodega = (
    integrado.loc[~integrado['bodega_encontrada'], 'categoria_producto']
    .value_counts(dropna=False).rename_axis('categoria_sin_bodega').to_frame('pedidos')
)
display(categorias_sin_bodega)

ruta_salida = S12_DIR / 'S12_PD_PallaresArbelaezSanchez_DatasetIntegrado.csv'
integrado.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f'Exportado: {ruta_salida} ({len(integrado):,} filas, {integrado.shape[1]} columnas)')

,valor
pedidos integrados,551790
ids de pedido únicos,551790
coincidencias CRM,366405
sin coincidencia CRM,185385
coincidencias catálogo bodega,459350
sin bodega asignada,92440
duplicados exactos integrados,0


,pedidos
categoria_sin_bodega,
Belleza,48686
belleza,17544
Belleza,8795
BELLEZA,8776
Belleza,8639


Exportado: C:\Users\L0s3r\OneDrive\Documentos\GitHub\pre_primera_clase\S12\S12_PD_PallaresArbelaezSanchez_DatasetIntegrado.csv (551,790 filas, 69 columnas)
Estado: PENDIENTE DE REVISIÓN INDEPENDIENTE
